<a href="https://colab.research.google.com/github/JanNogga/rl_ss25/blob/main/RL_Assignment_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Robot Learning

## Assignment 7

Solutions are due on 03.06.2025 before the lecture.

### Introduction

On this assignment sheet, we will use an environment from [OpenAi Gym](https://github.com/openai/gym). The setup requires some further packages. Since this installation is not trivial, and we could only test it in our setup, we strongly recommend that you execute the cells in this notebook in [Google Colab](https://research.google.com/colaboratory/). You should find a button which opens this file directly in Colab at the top of this notebook. If not, you can simply import the .ipynb manually.

If you have started your Colab session and are ready to proceed, uncomment the three lines in the code cell below. They will install everything required to simulate the Gym environment. If prompted to restart your runtime, do so, but you don't have to repeat the installation unless you delete your runtime.

**Warning: This is unlikely to work on your own computer, and might even mess up your system! Please only use the following lines in Colab. If you insist on using your own machine, please refer to installation instructions for Gym, torch and the box2d environments for your system.**

In [ ]:
# !apt-get -qq install xvfb x11-utils &> /dev/null
!pip install ufal.pybox2d --quiet &> /dev/null
!pip install pyvirtualdisplay moviepy pyglet PyOpenGL-accelerate --quiet &> /dev/null
!pip install numpy==1.23.5 matplotlib==3.7.0

The following cell imports packages required for the task.

In [ ]:
import numpy as np
import gym
import matplotlib.pyplot as plt
from pyvirtualdisplay import Display
from moviepy.editor import VideoClip
from moviepy.video.io.bindings import mplfig_to_npimage
import torch
import torch.nn as nn
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.11/dist-packages/moviepy/config_defaults.py:1: DeprecationWarning: invalid escape sequence '\P'
  """
/usr/local/lib/python3.11/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: DeprecationWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
  from scipy.ndimage.filters import sobel

  if event.key is 'enter':



## Task 7.1)

Your agent is in state $s_t$ and has the $Q$-values $Q(s_t,a) = [Q(s_t,a_0), Q(s_t,a_1), Q(s_t,a_2), Q(s_t,a_3)] = [5, 7, 3, 9]$. If the agent samples its action according to a probabilistic policy $\pi(s_t,a)$ which is created by softmax action selection from $Q(s_t,a)$, then what is the probability $Pr(a_3 | s_t)$ of taking action $a_3$ in state $s_t$?

<div style="text-align: right; font-weight:bold"> 3 Points </div>

In [ ]:
a = np.array([5,7,3,9])

a_3 = np.exp(a[3])/ np.sum(np.exp(a))
print("the prob of choosing action a3 :" , a_3)

the prob of choosing action a3 : 0.8649548767993755


Please answer in this text cell.


a3 = 9
by using probabilistic policy, the probability of choosing a3 at given state s is
86.5%



## Task 7.2)

In this task, we will combine an actor-critic method like on the previous sheet with a policy gradient algorithm to control an agent in a challenging environment: the Lunar Lander.

In [ ]:
# set up showing animations from the environment in Colab.
Display(visible=False).start()

Examine the code cell below. It has two distinct purposes:

* Showcase the agent-environment interaction for LunarLander-v2

* Show how you can capture frames from this environment to animate an episode afterwards.

Note that for training an agent in this environment, it is advisable to omit all code corresponding to the rendering. You can seperately render an episode of your agent's play afterwards.


In [22]:
# Name of the environment, if you are having problems you can switch to 'CartPole-v1', which is easier to solve.
ENV_NAME = 'LunarLander-v2' #'CartPole-v1'
# Dimension of the LunarLander state space. For 'CartPole-v1', use 4 instead
ENV_STATE_DIM =  4 # 4
# Lunar Lander has 4 discrete actions: [Do Nothing, Fire Left Booster, Fire Main Engine, Fire Right Booster], 'CartPole-v1' has 2
ENV_ACTION_DIM = 4 # 2
# If the agent reaches this score the task is seen as solved
SCORE_TO_SOLVE = 200 # 195
MAX_STEPS = None # 500
# Create the environment
env = gym.make(ENV_NAME)
# Reset the environment
state = env.reset() # state = [x, y, dx, dy, theta, dtheta, leg1_contact, leg2_contact]
# Track whether the episode is over
done = False
# List to append the frames produced by the environment renderer
frames = []
while not done:
  # Render current situation and append to frames
  frames.append(env.render('rgb_array'))
  # Select a random action
  action = env.action_space.sample()
  # Execute this action
  state, reward, done, info = env.step(action)
# Print the number of frames
print('Number of frames:', len(frames))
# Prevent the renderer from showing artifacts
plt.close()

Number of frames: 102


In [23]:
# Helper function to animate a list of frames as produced above
def visualize_trajectory(frames, fps=50):
  duration = int(len(frames) // fps + 1)
  fig, ax = plt.subplots()
  def make_frame(t, ind_max=len(frames)):
      ax.clear()
      ax.imshow(frames[min((int(fps*t),ind_max-1))])
      return mplfig_to_npimage(fig)
  plt.close()
  return VideoClip(make_frame, duration=duration)

In [24]:
# Get the animation from the frames of the played episode
animation = visualize_trajectory(frames)
# Show the animation
animation.ipython_display(fps=50, loop=True, autoplay=True)

t:   7%|▋         | 10/150 [01:24<00:14,  9.94it/s, now=None]

Moviepy - Building video __temp__.mp4.
Moviepy - Writing video __temp__.mp4




t:   7%|▋         | 10/150 [01:42<00:14,  9.94it/s, now=None]

Moviepy - Done !
Moviepy - video ready __temp__.mp4


Probably, the random agent will destroy itself instead of landing between the two flags. We would like you to improve upon this. Below, you are given a neural net with learnable weights $\theta$ which takes an environment state $s_t$ as input and can output either a state value $V_{\theta}(s_t)$ or a probability distribution over the actions $\pi_{\theta}(s_t,a)$.

In [ ]:
# If you feel like it, you can, but you do not need to adapt this
class DualNet(nn.Module):
    def __init__(self, state_dim=ENV_STATE_DIM, action_dim=ENV_ACTION_DIM, hidden_layer_dim=42):
        super(DualNet, self).__init__()
        # Create some layers to encode the input state
        self.layers = [nn.Linear(state_dim, hidden_layer_dim),
                       nn.LeakyReLU()]
        # Combine these layers into a net
        self.net = nn.Sequential(*self.layers)
        # Critic output layers to estimate V from the state encoding
        self.critic = nn.Sequential(*[nn.Linear(hidden_layer_dim,1)])
        # Actor output layers to estimate pi from the state encoding
        self.actor = nn.Sequential(*[nn.Linear(hidden_layer_dim,action_dim),
                                        nn.Softmax(dim=-1)])
    def forward(self, s, mode):
        # Convert input state to tensor
        device = next(self.net.parameters()).device
        x = torch.tensor(s).float().view(1,-1).to(device)
        # Encode state
        x = self.net(x)
        if mode == 'actor':
          # Return probability distribution over actions
          x = self.actor(x)
        else:
          # Return estimate of state value
          x = self.critic(x)
        return x.squeeze()

# Example usage:
# Create instance of the DualNet class
test_net = DualNet()
# Create a dummy state, round just for pretty printing
test_input = np.around(np.random.rand(ENV_STATE_DIM),2)
# Get the actor output
actor_out = test_net(test_input, mode='actor')
# Get the critic output
critic_out = test_net(test_input, mode='critic')
print('Dummy Input:', test_input)
print('Actor output:', actor_out)
print('Critic output:', critic_out)

Dummy Input: [0.86 0.9  0.21 0.18]
Actor output: tensor([0.1808, 0.2562, 0.2339, 0.3290], grad_fn=<SqueezeBackward0>)
Critic output: tensor(-0.2787, grad_fn=<SqueezeBackward0>)


Now to the task: Play episodes according to the following scheme:

* For each visited state $s_t$, store the output of the critic $V_\theta(s_t)$ in a list.

* Select an action $a_t$ by sampling from the distribution $\pi_{\theta}(s_t,a)$ output by the actor. In a list, store the log prob of the action: $l_t = log(\pi_{\theta}[s_t,a_t])$.

* Execute the action and observe the reward $r_{t+1}$ provided by the environment.

After each episode, use the stored rewards to calculate the Returns $R_t$ following each state $s_t$ using the discount factor $\gamma$. Next, calculate for each $t$

$$\delta_t = R_t - V_{\theta}(s_t)$$

Then, calculate the loss of the critic as

$$L_{critic} = 0.5 \sum_t \delta_t^2$$

and the loss of the actor using the log probs as

$$L_{actor} = \sum_t - l_t  \delta_t$$

and finally the total loss

$$L = L_{critic} + L_{actor}$$

Now, update the parameters $\theta$ using

$$\theta \leftarrow \theta + \alpha \nabla_{\theta}L$$

The Lunar Lander problem is considered solved when the agent achieves an average return of 200 over 100 episodes. Solve the problem, or play around 3000 episodes as described above and then report the average return of the final 100 episodes played. This means that you are welcome to preemptively stop training if the average return is sufficient. Then, play one more episode and animate it like in the example above. Use $\gamma = 0.99$ in your experiments. Save your final animation and place it into your sciebo folder along with this notebook. We recommend starting with a learning rate of $\alpha = 0.02$, but please note that this parameter might be sensitive to the specifics of your implementation. Solving the environment should not take longer than an hour, but we have seen implementations that completed the task in as little as 10 minutes.

### Hints:

Following tips might help you complete this task:

* You might need to convert your return $R_t$ to the correct datatype:

$$\delta_t = torch.tensor(R_t) - V_{\theta}(s_t)$$

* If you want to use numpy to sample from $\pi_{\theta}(s_t,a)$, you can get a numpy array by calling

$$\pi_{\theta}(s_t,a).detach().numpy()$$

* When you calculate the log probs $l_t$, preserve the torch gradient graph by using the torch function

$$l_t = torch.log(\pi_{\theta}[s_t,a_t])$$

* When you calculate $L_{actor}$, use $\delta_t.item()$ to ensure that the actor's loss does not directly influence the critic's gradients.

* It might be easier to solve the task for the Cart-Pole environment first, just change ENV_NAME, ENV_STATE_DIM and ENV_ACTION_DIM in one of the previous code blocks.

* Standardizing the Returns (zero mean and std 1) before calculating $\delta_t$ can boost performance.

* Below you are already given a rough structure for the algorithm. If you stick to it, torch will compute and apply $\nabla_{\theta}L$ for you!

<div style="text-align: right; font-weight:bold"> 10 +  3 (animation) Points </div>

In [ ]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set the correct state dimension for Lunar Lander
ENV_STATE_DIM = 8 # Lunar Lander state dimension
ENV_ACTION_DIM = 4 # Lunar Lander action dimension

net = DualNet(state_dim=ENV_STATE_DIM, action_dim=ENV_ACTION_DIM).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.02)
GAMMA = 0.99
scores = []

for episode in range(3000):
    state = env.reset()
    log_probs, values, rewards = [], [], []
    done = False

    while not done:

        # state_tensor = torch.tensor(state).float().unsqueeze(0).to(device)

        # Actor: get probabilities and sample action
        # Pass the numpy array directly to the net's forward method
        probs = net(state, mode='actor')
        action = np.random.choice(ENV_ACTION_DIM, p=probs.detach().cpu().numpy())
        log_prob = torch.log(probs[action])

        # Critic: estimate state value
        # Pass the numpy array directly to the net's forward method
        value = net(state, mode='critic')

        # Env step
        next_state, reward, done, _ = env.step(action)

        # Save
        log_probs.append(log_prob)
        values.append(value)
        rewards.append(reward)

        state = next_state

    # Compute returns and advantages
    returns = []
    R = 0
    for r in reversed(rewards):
        R = r + GAMMA * R
        returns.insert(0, R)

    returns = torch.tensor(returns).float().to(device)
    values = torch.stack(values)
    # Use .squeeze() to ensure values tensor has correct shape
    deltas = returns - values.squeeze()
    deltas = (deltas - deltas.mean()) / (deltas.std() + 1e-8)

    # Compute losses
    # Used .item() as suggested in hints
    actor_loss = -torch.stack(log_probs) * deltas.detach()
    critic_loss = 0.5 * deltas.pow(2)
    loss = actor_loss.sum() + critic_loss.sum()

    # Back propogation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_reward = sum(rewards)
    scores.append(total_reward)
    # avg_score = np.mean(scores[-100:])

    # print(f"Episode {episode+1}, Return: {total_reward:.1f}, Avg100: {avg_score:.1f}")

    # if avg_score >= 200 and episode >= 100:
    #     print(" Environment Solved!")
    #     break

    if (episode + 1) % 100 == 0:
        avg_last_100 = np.mean(scores[-100:])
        print(f"Episode {episode+1}, Average Return (last 100): {avg_last_100:.2f}")

# Rollout and render after training (or when solved)
state = env.reset()
done = False
frames = []

while not done:
    frames.append(env.render(mode='rgb_array'))
    # Pass the numpy array directly to the net's forward method
    probs = net(state, mode='actor')
    action = np.random.choice(ENV_ACTION_DIM, p=probs.detach().cpu().numpy()) # Move to CPU before numpy conversion
    state, _, done, _ = env.step(action)
plt.close()

Episode 100, Average Return (last 100): -204.02
Episode 200, Average Return (last 100): -134.09
Episode 300, Average Return (last 100): -130.47
Episode 400, Average Return (last 100): -129.41
Episode 500, Average Return (last 100): -131.96
Episode 600, Average Return (last 100): -129.06
Episode 700, Average Return (last 100): -123.34
Episode 800, Average Return (last 100): -136.14
Episode 900, Average Return (last 100): -132.76
Episode 1000, Average Return (last 100): -124.76
Episode 1100, Average Return (last 100): -125.51
Episode 1200, Average Return (last 100): -128.46
Episode 1300, Average Return (last 100): -130.57
Episode 1400, Average Return (last 100): -127.28
Episode 1500, Average Return (last 100): -125.65
Episode 1600, Average Return (last 100): -124.21
Episode 1700, Average Return (last 100): -122.76
Episode 1800, Average Return (last 100): -128.86
Episode 1900, Average Return (last 100): -120.77
Episode 2000, Average Return (last 100): -121.59
Episode 2100, Average Return 

In [ ]:
# Helper function to animate a list of frames as produced above
def visualize_trajectory(frames, fps=50):
  duration = int(len(frames) // fps + 1)
  fig, ax = plt.subplots()
  def make_frame(t, ind_max=len(frames)):
      ax.clear()
      ax.imshow(frames[min((int(fps*t),ind_max-1))])
      return mplfig_to_npimage(fig)
  plt.close()
  # Create the VideoClip and explicitly set its fps attribute
  clip = VideoClip(make_frame, duration=duration)
  clip.fps = fps # Set the fps attribute here
  return clip

In [ ]:
# Animate
animation = visualize_trajectory(frames)
# You might want to save to a specific path if running in Colab, e.g., "/content/lunar_lander.mp4"
animation.write_videofile("lunar_lander.mp4", fps=50)
animation.ipython_display()

Moviepy - Building video lunar_lander.mp4.
Moviepy - Writing video lunar_lander.mp4



Moviepy - Done !
Moviepy - video ready lunar_lander.mp4
Moviepy - Building video __temp__.mp4.
Moviepy - Writing video __temp__.mp4



Moviepy - Done !
Moviepy - video ready __temp__.mp4


## Task 7.3)

In the previous task, you calculate the loss of the policy network as

$$L_{actor} = \sum_t - log(\pi_{\theta}[s_t,a_t])  \delta_t$$

with

$$\delta_t = R_t - V_{\theta}(s_t).$$

Give an intuitive explanation why minimizing this term leads to actions with a good outcome being more likely and actions with a bad outcome becoming less likely.

<div style="text-align: right; font-weight:bold"> 4 Points </div>

Please answer in this text cell.

When $\delta_t = R_t - V_\theta(s_t)$ is positive, it means that the actual return $R_t$ received after taking action $a_t$ in state $s_t$ is better than the critic's current estimate of the value of state $s_t$, $V_\theta(s_t)$. This indicates that the action $a_t$ at state $s_t$ was "better than expected".

The actor's loss is $L_{actor} = \sum_t - log(\pi_{\theta}[s_t,a_t]) \delta_t$.
Minimizing this loss means we want to decrease the value of $L_{actor}$.


If $\delta_t > 0$ (good outcome):
We are minimizing $- log(\pi_{\theta}[s_t,a_t]) \times (\text{+ value})$.
This is equivalent to minimizing a negative value, which means we want to make $- log(\pi_{\theta}[s_t,a_t])$ as negative as possible.
$log(\pi_{\theta}[s_t,a_t])$ becomes more positive.
Since the logarithm is an increasing function, a more positive $log(\pi_{\theta}[s_t,a_t])$ means a higher probability $\pi_{\theta}[s_t,a_t]$.
So, when an action leads to a better expected outcome ($\delta_t > 0$), the policy will taake that action more likely for that state.

If $\delta_t < 0$ (bad outcome):
We are minimizing $- log(\pi_{\theta}[s_t,a_t]) \times (\text{negative value})$.
This is equivalent to minimizing a positive value.

A smaller $log(\pi_{\theta}[s_t,a_t])$ means a lower probability $\pi_{\theta}[s_t,a_t]$.
So, when an action leads to a worse expected outcome ($\delta_t < 0$), the policy is updated to make that action less likely in that state.

In summary, the term $- log(\pi_{\theta}[s_t,a_t])$ is large and positive when $\pi_{\theta}[s_t,a_t]$ is small, and small (negative) when $\pi_{\theta}[s_t,a_t]$ is large. Multiplying this by $\delta_t$:
- If $\delta_t$ is positive, minimizing the loss requires making $- log(\pi_{\theta}[s_t,a_t])$ small, thus increasing $\pi_{\theta}[s_t,a_t]$.
- If $\delta_t$ is negative, minimizing the loss requires making $- log(\pi_{\theta}[s_t,a_t])$ large, thus decreasing $\pi_{\theta}[s_t,a_t]$.

policy gradient will take actions that result in positive advantage ($\delta_t > 0$) and donot take actions that result in negative advantage ($\delta_t < 0$).